
# Clasificador OptDigits en **PyTorch** (Google Colab-Ready)

Este notebook entrena una **RNA/MLP** para el dataset **OptDigits** (8×8 pixeles, 10 clases) usando PyTorch.  
Incluye:
- Carga y preprocesamiento de datos.
- Modelo `MLP` equivalente a `Flatten → Dense(128, ReLU) → Dense(64, ReLU) → Dense(10)`.
- Entrenamiento y evaluación con `CrossEntropyLoss` y `Adam`.
- Gráficas de **accuracy** y **loss**.
- (Opcional) **Aprendizaje incremental por bloques** (simulado) para continuar el entrenamiento sin re-inicializar pesos.

In [ ]:

# Comando para instalar torch, torchbision scikit-learrning y matplotlib, pero por lo general no es necesario
# !pip install -q torch torchvision scikit-learn matplotlib


## 1) Importaciones y dispositivo

In [ ]:

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import TensorDataset, DataLoader
from sklearn.datasets import load_digits
from sklearn.model_selection import train_test_split
import matplotlib.pyplot as plt

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Dispositivo:", device)


## 2) Carga y preprocesamiento de datos

In [ ]:

digits = load_digits()
images = digits.images.astype(np.float32) / 16.0
labels = digits.target.astype(np.int64)

X_train, X_test, y_train, y_test = train_test_split(
    images, labels, test_size=0.2, random_state=42, stratify=labels
)

X_train = torch.tensor(X_train, dtype=torch.float32)
X_test  = torch.tensor(X_test,  dtype=torch.float32)
y_train = torch.tensor(y_train, dtype=torch.long)
y_test  = torch.tensor(y_test,  dtype=torch.long)

train_ds = TensorDataset(X_train, y_train)
test_ds  = TensorDataset(X_test,  y_test)
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)
test_dl  = DataLoader(test_ds,  batch_size=32, shuffle=False)

print(f"Train: {len(train_ds)} | Test: {len(test_ds)}")


## 3) Modelo MLP

In [ ]:

class MLP(nn.Module):
    def __init__(self):
        super().__init__()
        self.flatten = nn.Flatten()
        self.fc1 = nn.Linear(64, 128)
        self.fc2 = nn.Linear(128, 64)
        self.fc3 = nn.Linear(64, 10)
    def forward(self, x):
        x = self.flatten(x)
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        return self.fc3(x)

model = MLP().to(device)
print(model)


## 4) Pérdida, optimizador y helpers

In [ ]:

criterion = nn.CrossEntropyLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

def train_one_epoch(model, dl, opt, crit, device):
    model.train()
    total, correct, run_loss = 0, 0, 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        opt.zero_grad()
        logits = model(xb)
        loss = crit(logits, yb)
        loss.backward()
        opt.step()
        run_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return run_loss/total, correct/total

@torch.no_grad()
def evaluate(model, dl, crit, device):
    model.eval()
    total, correct, run_loss = 0, 0, 0.0
    for xb, yb in dl:
        xb, yb = xb.to(device), yb.to(device)
        logits = model(xb)
        loss = crit(logits, yb)
        run_loss += loss.item() * xb.size(0)
        pred = logits.argmax(dim=1)
        correct += (pred == yb).sum().item()
        total += xb.size(0)
    return run_loss/total, correct/total


## 5) Entrenamiento

In [ ]:

epochs = 10
hist_train_loss, hist_train_acc = [], []
hist_val_loss, hist_val_acc = [], []

for ep in range(1, epochs+1):
    tr_loss, tr_acc = train_one_epoch(model, train_dl, optimizer, criterion, device)
    va_loss, va_acc = evaluate(model, test_dl, criterion, device)
    hist_train_loss.append(tr_loss); hist_train_acc.append(tr_acc)
    hist_val_loss.append(va_loss);   hist_val_acc.append(va_acc)
    print(f"Época {ep:02d} | Train Loss {tr_loss:.4f} Acc {tr_acc:.4f} | Val Loss {va_loss:.4f} Acc {va_acc:.4f}")


## 6) Gráficas

In [ ]:

plt.figure()
plt.plot(hist_train_loss, label="Train Loss")
plt.plot(hist_val_loss, label="Val Loss")
plt.xlabel("Época"); plt.ylabel("Loss"); plt.title("Evolución de Loss"); plt.legend(); plt.show()

plt.figure()
plt.plot(hist_train_acc, label="Train Acc")
plt.plot(hist_val_acc, label="Val Acc")
plt.xlabel("Época"); plt.ylabel("Accuracy"); plt.title("Evolución de Accuracy"); plt.legend(); plt.show()


## 7) (Opcional) Incremental por bloques

In [ ]:

extra_blocks = 0   # cámbialo a 6 si quieres simular 6 bloques incrementales
epochs_per_block = 2

block_hist = []  # guardará (block_idx, train_loss, train_acc, val_loss, val_acc)

for b in range(1, extra_blocks + 1):
    for _ in range(epochs_per_block):
        _ = train_one_epoch(model, train_dl, optimizer, criterion, device)
    t_loss, t_acc = evaluate(model, train_dl, criterion, device)
    v_loss, v_acc = evaluate(model, test_dl, criterion, device)
    block_hist.append((b, t_loss, t_acc, v_loss, v_acc))
    print(f"[Bloque {b}] Train Loss: {t_loss:.4f} Acc: {t_acc:.4f} | Val Loss: {v_loss:.4f} Acc: {v_acc:.4f}")


###  7.1) Gráficas por bloque (si `extra_blocks > 0`)



In [ ]:
if len(block_hist) > 0:
    blocks = [b for (b, *_ ) in block_hist]
    t_losses = [tl for (_, tl, *_ ) in block_hist]
    t_accs   = [ta for (_, _, ta, *_ ) in block_hist]
    v_losses = [vl for (*_, vl, _ ) in block_hist]
    v_accs   = [va for (*_, _, va) in block_hist]

    plt.figure()
    plt.plot(blocks, t_losses, marker="o", label="Train Loss")
    plt.plot(blocks, v_losses, marker="o", label="Val Loss")
    plt.xlabel("Bloque incremental")
    plt.ylabel("Loss")
    plt.title("Loss por bloque (incremental)")
    plt.legend()
    plt.show()

    plt.figure()
    plt.plot(blocks, t_accs, marker="o", label="Train Acc")
    plt.plot(blocks, v_accs, marker="o", label="Val Acc")
    plt.xlabel("Bloque incremental")
    plt.ylabel("Accuracy")
    plt.title("Accuracy por bloque (incremental)")
    plt.legend()
    plt.show()
else:
    print("No se configuraron bloques incrementales (extra_blocks = 0).")

## 8) Evaluación final

In [ ]:

test_loss, test_acc = evaluate(model, test_dl, criterion, device)
print(f"Resultado final -> Test Loss: {test_loss:.4f} | Test Acc: {test_acc:.4f}")
